# 用训练得到的模型预测-视频

同济子豪兄 2023-7-13

## 进入mmsegmentation目录

In [8]:
# import os
# os.chdir('mmsegmentation')

import os
# 假设 mmsegmentation 的绝对路径是 /project/mmsegmentation
mmsegmentation_path = "E:/bishe_custom_data/mmsegmentation-main"
# 切换到 mmsegmentation 文件夹
os.chdir(mmsegmentation_path)
# 验证当前工作目录
print("当前工作目录:", os.getcwd())  # 输出: /project/mmsegmentation

当前工作目录: E:\bishe_custom_data\mmsegmentation-main


## MMSegmentation官方视频/摄像头预测

In [2]:
# !python3 demo/video_demo.py \
#          data/video_watermelon_3.mov \
#          Zihao-Configs/ZihaoDataset_KNet_20230712.py \
#          checkpoint/Zihao_KNet.pth \
#          --device cpu \
#          --opacity 0.5 \
#          --show

## OpenCV

### 导入工具包

In [9]:
import time
import numpy as np
from tqdm import tqdm
import cv2

import mmcv
from mmseg.apis import init_model, inference_model

### 载入训练好的模型

In [10]:
# 模型 config 配置文件
config_file = 'Zihao-Configs/ZihaoDataset_DeepLabV3plus_20230818.py'

# 模型 checkpoint 权重文件
checkpoint_file = 'work_dirs/ZihaoDataset-DeepLabV3plus/best_mIoU_iter_5300.pth'

# device = 'cpu'
device = 'cuda:0'

In [11]:
model = init_model(config_file, checkpoint_file, device=device)

E:\bishe_custom_data\mmsegmentation-main\mmseg\models\losses\cross_entropy_loss.py:250: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  warnings.warn(


Loads checkpoint by local backend from path: work_dirs/ZihaoDataset-DeepLabV3plus/best_mIoU_iter_5300.pth


### 各类别的配色方案（BGR）

In [12]:
# palette = [
#     ['background', [127,127,127]],
#     ['red', [0,0,200]],
#     ['green', [0,200,0]],
#     ['white', [144,238,144]],
#     ['seed-black', [30,30,30]],
#     ['seed-white', [8,189,251]]
# ]

# 自定义 井下数据集 5 个类别的配色方案（BGR 格式）
palette = [
    ['background', [200, 150, 150]],  # 背景
    ['person', [0, 0, 255]],          # 人（红色）
    ['roadheader', [0, 255, 0]],      # roadheader（绿色）
    ['robot', [0, 255, 255]],         # 机器人（黄色）
    ['shearer', [255, 0, 0]],         # shearer（蓝色）
    # 可以继续添加其他类别
]

In [13]:
palette_dict = {}
for idx, each in enumerate(palette):
    palette_dict[idx] = each[1]

In [14]:
palette_dict

{0: [200, 150, 150],
 1: [0, 0, 255],
 2: [0, 255, 0],
 3: [0, 255, 255],
 4: [255, 0, 0]}

### 逐帧处理函数

In [15]:
opacity = 0.3 # 透明度，越大越接近原图

In [16]:
def process_frame(img_bgr):
    
    # 记录该帧开始处理的时间
    start_time = time.time()
    
    # 语义分割预测
    result = inference_model(model, img_bgr)
    pred_mask = result.pred_sem_seg.data[0].cpu().numpy()
    
    # 将预测的整数ID，映射为对应类别的颜色
    pred_mask_bgr = np.zeros((pred_mask.shape[0], pred_mask.shape[1], 3))
    for idx in palette_dict.keys():
        pred_mask_bgr[np.where(pred_mask==idx)] = palette_dict[idx]
    pred_mask_bgr = pred_mask_bgr.astype('uint8')
    
    # 将语义分割预测图和原图叠加显示
    pred_viz = cv2.addWeighted(img_bgr, opacity, pred_mask_bgr, 1-opacity, 0)

    return pred_viz

### 视频逐帧处理

In [17]:
# 视频逐帧处理代码模板
# 不需修改任何代码，只需定义process_frame函数即可
# 同济子豪兄 2021-7-10

def generate_video(input_path='videos/robot.mp4'):
    filehead = input_path.split('/')[-1]
    output_path = "out-" + filehead
    
    print('视频开始处理',input_path)
    
    # 获取视频总帧数
    cap = cv2.VideoCapture(input_path)
    frame_count = 0
    while(cap.isOpened()):
        success, frame = cap.read()
        frame_count += 1
        if not success:
            break
    cap.release()
    print('视频总帧数为',frame_count)
    
    # cv2.namedWindow('Crack Detection and Measurement Video Processing')
    cap = cv2.VideoCapture(input_path)
    frame_size = (cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))
    # fourcc = cv2.VideoWriter_fourcc(*'XVID')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = cap.get(cv2.CAP_PROP_FPS)

    out = cv2.VideoWriter(output_path, fourcc, fps, (int(frame_size[0]), int(frame_size[1])))
    
    # 进度条绑定视频总帧数
    with tqdm(total=frame_count-1) as pbar:
        try:
            while(cap.isOpened()):
                success, frame = cap.read()
                if not success:
                    break

                # 处理帧
                # frame_path = './temp_frame.png'
                # cv2.imwrite(frame_path, frame)
                try:
                    frame = process_frame(frame)
                except:
                    print('报错！', error)
                    pass
                
                if success == True:
                    # cv2.imshow('Video Processing', frame)
                    out.write(frame)

                    # 进度条更新一帧
                    pbar.update(1)

                # if cv2.waitKey(1) & 0xFF == ord('q'):
                    # break
        except:
            print('中途中断')
            pass

    cv2.destroyAllWindows()
    out.release()
    cap.release()
    print('视频已保存', output_path)

In [18]:
generate_video(input_path='data/robot.mp4')
#视频直接放在mmsegmentation-main/ 下面

视频开始处理 data/robot.mp4
视频总帧数为 1665


100%|██████████| 1664/1664 [05:45<00:00,  4.81it/s]

视频已保存 out-robot.mp4
